## Creating RAG Application using typesense 

In [20]:
import os
import typesense
from dotenv import load_dotenv
from pydantic import SecretStr

load_dotenv()



True

In [21]:
search_key = os.getenv("TYPESENSE_SEARCHONLY_KEY")
admin_key = os.getenv("TYPESENSE_ADMIN_KEY")

if not search_key:
    raise ValueError("Typesense search-only key was not loaded")

if not admin_key:
    raise ValueError("Typesense admin key was not loaded")

client = typesense.Client({
    "nodes": [{
        "host": "o0r68mv2bgxjpi5zp-1.a2.typesense.net",
        "port": 443,
        "protocol": "https",
    }],
    "api_key": admin_key,
    "connection_timeout_seconds": 5,
})
# print(client)
# print(client.collections.retrieve())

### Schema design

In [22]:
from typesense.types.collection import CollectionCreateSchema

book_schema: CollectionCreateSchema = {
    'name': 'books',
    'fields':[
        {'name':'title','type':'string'},
        {'name':'authors','type':'string[]', 'facet':True},
        {'name':'publication_year', 'type':'int32','facet':True},
        {'name':'ratings_count', 'type':'int32'},
        {'name':'average_rating','type':'float'}
    ],
    'default_sorting_field':'ratings_count'
}
# Create the collection only if it does not exist
try:
    collection_info = client.collections["books"].retrieve()
    print("Collection already exists.")
except Exception as exc:
    # Do not treat every exception as a missing collection.
    # Authentication and network failures must not be ignored.
    if getattr(exc, "http_status", None) == 404:
        collection_info = client.collections.create(book_schema)
        print("Collection created successfully.")
    else:
        raise
print(collection_info)

Collection already exists.
{'created_at': 1792566386, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'

### Import data into Schema

In [ ]:
from pathlib import Path
import json 
jsonl_path = Path("books.jsonl")

with jsonl_path.open('r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    import_results = client.collections['books'].documents.import_(data)

    # Check individual document import results
for line_number, result in enumerate(import_results.splitlines(), start=1):
    item = json.loads(result)
    if not item.get("success"):
        print(f"Import failed on line {line_number}: {item}")


In [ ]:
search_parameters = {
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':'ratings_count:desc'
}

client.collections['books'].documents.search(search_parameters) # type: ignore

### Using Langchain + tyoesense + Groq LLM + RAG application 

In [27]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [37]:
GROQ_KEY = os.environ["GROQ_API_KEY"]
GROQ_KEY = SecretStr(GROQ_KEY)

loader = TextLoader("./Data/text_files/AI.txt")
document = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents=document)
embeddings = HuggingFaceEmbeddings()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 731.32it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

#### Create the Vector store using Typesense 


In [39]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "o0r68mv2bgxjpi5zp-1.a2.typesense.net",
        "port": 443,
        "protocol": "https",
        "typesense_api_key": admin_key,
        "typesense_collection_name": "lang-chain",
    },
)

In [41]:
query = "what are the contributions of AI?"
found_docs = docsearch.similarity_search(query=query)
print(found_docs[0].page_content)

ESSAY ON: ARTIFICIAL INTELLIGENCE â€” TRANSFORMING THE FUTURE OF HUMANITY

Artificial Intelligence (AI) is one of the most influential technologies of the modern age. It refers to the ability of computer systems to perform tasks that usually require human intelligence, such as learning, reasoning, understanding language, recognizing images, and solving problems. From virtual assistants and recommendation systems to medical research and self-driving vehicles, AI is changing how people live, learn, and work. Its development creates exciting opportunities, but it also raises important questions about responsibility, fairness, and the future of humanity.


In [51]:
### Retriever 
retriever = docsearch.as_retriever()

In [52]:

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("Set the GROQ_API_KEY environment variable before running this cell.")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3,
    api_key=SecretStr(api_key),
)
question = "what is AI ethics ?"
# Generate answers 
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    document.page_content
    for document in retrieved_docs
    if getattr(document, "page_content", None)
)

prompt = """Use the following context to answer the question concisely.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt.format(context=context, question=question))
answer = response.content
print(answer) 
response = llm.invoke([prompt.format(context=context, question=question)])
answer = response.content

AI ethics is the set of principles and guidelines that govern the responsible design, deployment, and use of artificial intelligence. It focuses on ensuring AI systems are safe, reliable, fair, and transparent; protect personal data; avoid bias and discrimination; allow human oversight—especially in critical areas like healthcare, education, employment, and law; and enable people to understand and challenge AI‑driven decisions. The goal is to align AI development with human values such as empathy, dignity, responsibility, and compassion, so that technology serves to improve human life rather than replace it.
